<a href="https://colab.research.google.com/github/infinityrohit07/tensorflow-learning/blob/main/notebooks/17_phase2_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Classification Loss Functions

### What is Binary Cross-Entropy?

Binary Cross-Entropy (BCE) is a loss function used in binary classification to measure how well a model's predicted probabilities match the actual binary labels (0 or 1).

- Lower loss = better prediction.
- Higher loss = worse prediction.
- Confident but incorrect predictions receive a high penalty.

#### **Understanding BCE with examples**

Suppose the actual label is `1` (Spam).

| Actual label | Predicted probability | Result |
|---|---:|---|
| 1 | 0.90 | Good prediction |
| 1 | 0.60 | Less confident |
| 1 | 0.10 | Bad prediction |

<br>

If the actual label is `0` (Not Spam):

| Actual label | Predicted probability | Result |
|---|---:|---|
| 0 | 0.10 | Good prediction |
| 0 | 0.90 | Bad prediction |

**Remember:**
- Actual label `1` → probability should be close to `1`.
- Actual label `0` → probability should be close to `0`.

#### **BCE Formula**

$$
L=-[y\log(p)+(1-y)\log(1-p)]
$$

Where:

- $(y)$ = Actual label (0 or 1).
- $(p)$ = Predicted probability.
- $(\log)$ = Natural logarithm.

<br>

The formula has two cases:

**When actual label is 1:**

$$
L=-\log(p)
$$

**When actual label is 0:**

$$
L=-\log(1-p)
$$

#### **Implementing BCE using TensorFlow**

In [ ]:
import tensorflow as tf

# Actual labels
y_true = tf.constant([1, 0, 1, 0], dtype=tf.float32)

# Predicted probabilities
y_pred = tf.constant([0.90, 0.10, 0.80, 0.20], dtype=tf.float32)

# Create BCE loss function
bce = tf.keras.losses.BinaryCrossentropy()

# Calculate loss
loss = bce(y_true, y_pred)

print(loss.numpy())

0.16425204


#### **BCE and Sigmoid Activation**

For binary classification, we commonly use a Dense output layer with one neuron and sigmoid activation:

```python
tf.keras.layers.Dense(1, activation="sigmoid")
```

When using sigmoid probabilities:

```python
tf.keras.losses.BinaryCrossentropy(from_logits=False)
```

If the model outputs raw logits without sigmoid:

```python
tf.keras.losses.BinaryCrossentropy(from_logits=True)
```

**Important:** Do not apply sigmoid to the output when using `from_logits=True`. The loss function handles the logits internally.

#### **Observing BCE Loss with Different Predictions**

In [ ]:
import tensorflow as tf

# Actual label
y_true = tf.constant([1.0])

# Different predicted probabilities
predictions = [0.50, 0.70, 0.90, 0.99]

# Create BCE loss function
bce = tf.keras.losses.BinaryCrossentropy()

# Calculate loss for each prediction
for prediction in predictions:
    y_pred = tf.constant([prediction])
    loss = bce(y_true, y_pred)

    print(f"Prediction: {prediction} | Loss: {loss.numpy():.4f}")

Prediction: 0.5 | Loss: 0.6931
Prediction: 0.7 | Loss: 0.3567
Prediction: 0.9 | Loss: 0.1054
Prediction: 0.99 | Loss: 0.0101


As the predicted probability gets closer to the actual label `1`, the BCE loss decreases.

#### **Key Takeaways**

- BCE is used for binary classification.
- It measures the difference between actual labels and predicted probabilities.
- Lower BCE loss indicates better predictions.
- Confident incorrect predictions are heavily penalized.
- Sigmoid probabilities use `from_logits=False`.
- Raw logits use `from_logits=True`.

### What is Categorical Cross-Entropy?

Categorical Cross-Entropy (CCE) is a loss function used in multiclass classification to measure how well a model's predicted probabilities match the actual class labels.

- Lower loss = better prediction.
- Higher loss = worse prediction.
- It is commonly used with Softmax activation.
- It is used with one-hot encoded labels.

#### **Understanding CCE with an Example**

Suppose we classify images into three categories:

- Cat = 0
- Dog = 1
- Horse = 2

The actual image is a Dog.

<br>

**Actual label (One-Hot Encoding):**

```python
y_true = [0, 1, 0]
```

**Predicted probabilities (Softmax):**

```python
y_pred = [0.10, 0.80, 0.10]
```
<br>

| Class | Actual | Predicted |
|---|---:|---:|
| Cat | 0 | 0.10 |
| Dog | 1 | 0.80 |
| Horse | 0 | 0.10 |

The model assigns 80% probability to Dog.

#### **CCE Formula**

$$
L=-\sum_{i=1}^{N}y_i\log(p_i)
$$

Where:

- $(y_i)$ = Actual one-hot encoded label.
- $(p_i)$ = Predicted probability for each class.
- $(N)$ = Number of classes.

Since one-hot encoding contains only one `1`, only the probability assigned to the correct class contributes to the loss.

For example:

$$
L=-\log(0.80)
$$

$$
L\approx0.2231
$$

#### **Understanding Prediction Quality**

Suppose the actual class is Dog.

| Dog probability | CCE loss |
|---:|---:|
| 0.20 | 1.6094 |
| 0.50 | 0.6931 |
| 0.80 | 0.2231 |
| 0.95 | 0.0513 |

As the probability assigned to the correct class increases, CCE loss decreases.

**Important:** CCE considers the probability assigned to the actual class, not just whether the predicted class is correct.

#### **Implementing CCE Using TensorFlow**

In [ ]:
import tensorflow as tf

# Actual labels (one-hot encoded)
y_true = tf.constant([
    [1, 0, 0],  # Cat
    [0, 1, 0],  # Dog
    [0, 0, 1]   # Horse
], dtype=tf.float32)

# Model predictions (Softmax probabilities)
y_pred = tf.constant([
    [0.80, 0.10, 0.10],
    [0.10, 0.70, 0.20],
    [0.10, 0.20, 0.70]
], dtype=tf.float32)

# Create Categorical Cross-Entropy loss
cce = tf.keras.losses.CategoricalCrossentropy()

# Calculate loss
loss = cce(y_true, y_pred)

print(loss.numpy())

0.3121645


#### **How Does TensorFlow Calculate the Loss?**

TensorFlow calculates the individual loss for each sample.

| Sample | Actual class | Correct-class probability | Loss |
|---|---|---:|---:|
| 1 | Cat | 0.80 | 0.2231 |
| 2 | Dog | 0.70 | 0.3567 |
| 3 | Horse | 0.70 | 0.3567 |

It then averages the individual losses:

$$
L=\frac{0.2231+0.3567+0.3567}{3}
$$

$$
L\approx0.3122
$$

#### **Key Takeaways**

- CCE is used for multiclass classification.
- It commonly works with Softmax activation.
- It uses one-hot encoded labels.
- It measures the probability assigned to the correct class.
- Lower CCE loss means a better prediction.
- TensorFlow averages the individual sample losses by default.

### What is Sparse Categorical Cross-Entropy?

Sparse Categorical Cross-Entropy (SCCE) is a loss function used in multiclass classification when actual labels are represented as integer class IDs instead of one-hot encoded vectors.

- Lower loss = better prediction.
- Higher loss = worse prediction.
- Commonly used with Softmax activation.
- Uses integer-encoded labels.

#### **Understanding SCCE with an Example**

Suppose we classify images into three categories:

- Cat = 0
- Dog = 1
- Horse = 2

**Actual labels (Integer Encoding):**

```python
y_true = [0, 1, 2]
```

**Predicted probabilities (Softmax):**

```python
y_pred = [
    [0.80, 0.10, 0.10],  # Cat
    [0.10, 0.70, 0.20],  # Dog
    [0.10, 0.20, 0.70]   # Horse
]
```

Each row represents one sample, and each column represents a class.

#### **SCCE Formula**

$$
L=-\log(p_{\text{correct class}})
$$

Where:

- $(p_{\text{correct class}})$ = Predicted probability assigned to the actual class.

<br>

For example, if the actual class is Dog (class ID 1) and its predicted probability is 0.70:

$$
L=-\log(0.70)
$$

$$
L\approx0.3567
$$

#### **CCE vs SCCE**

Both loss functions calculate the same type of multiclass classification loss. The main difference is how actual labels are represented.

<br>

**1. Categorical Cross-Entropy (CCE):**

Uses one-hot encoded labels.

```python
y_true = [
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
]
```

```python
tf.keras.losses.CategoricalCrossentropy()
```

<br>

**2. Sparse Categorical Cross-Entropy (SCCE):**

Uses integer-encoded labels.

```python
y_true = [0, 1, 2]
```

```python
tf.keras.losses.SparseCategoricalCrossentropy()
```

#### **Implementing SCCE Using TensorFlow**

In [ ]:
import tensorflow as tf

# Actual labels (integer encoded)
y_true = tf.constant([0, 1, 2], dtype=tf.int32)

# Predicted probabilities
y_pred = tf.constant([
    [0.80, 0.10, 0.10],  # Cat
    [0.10, 0.70, 0.20],  # Dog
    [0.10, 0.20, 0.70]   # Horse
], dtype=tf.float32)

# Create SCCE loss function
scce = tf.keras.losses.SparseCategoricalCrossentropy()

# Calculate loss
loss = scce(y_true, y_pred)

print(loss.numpy())

0.31216446


#### **How Does TensorFlow Calculate the Loss?**

TensorFlow calculates the individual loss for each sample.

| Sample | Actual class | Correct-class probability | Loss |
|---|---|---:|---:|
| 1 | Cat | 0.80 | 0.2231 |
| 2 | Dog | 0.70 | 0.3567 |
| 3 | Horse | 0.70 | 0.3567 |

It then averages the individual losses:

$$
L=\frac{0.2231+0.3567+0.3567}{3}
$$

$$
L\approx0.3122
$$

#### **Key Takeaways**

- SCCE is used for multiclass classification.
- It uses integer-encoded labels.
- It commonly works with Softmax activation.
- It measures the probability assigned to the correct class.
- Lower SCCE loss means a better prediction.
- CCE and SCCE calculate the same type of loss but accept different label formats.

### Choosing Based on Classification Type

| Classification type | Loss function |
|---|---|
| Binary classification | Binary Cross-Entropy (BCE) |
| Multiclass with one-hot labels | Categorical Cross-Entropy (CCE) |
| Multiclass with integer labels | Sparse Categorical Cross-Entropy (SCCE) |

### Choosing Based on Output Activation

| Loss function | Common output activation |
|---|---|
| Binary Cross-Entropy | Sigmoid |
| Categorical Cross-Entropy | Softmax |
| Sparse Categorical Cross-Entropy | Softmax |

- Sigmoid is commonly used for binary classification.
- Softmax is commonly used for multiclass classification.
- Loss functions measure how well predicted probabilities match actual labels.

### TensorFlow Examples

#### 1. Binary Classification

```python
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, activation="sigmoid")
])

loss_fn = tf.keras.losses.BinaryCrossentropy()
```

#### 2. Multiclass with One-Hot Labels

```python
model = tf.keras.Sequential([
    tf.keras.layers.Dense(3, activation="softmax")
])

loss_fn = tf.keras.losses.CategoricalCrossentropy()
```

#### 3. Multiclass with Integer Labels

```python
model = tf.keras.Sequential([
    tf.keras.layers.Dense(3, activation="softmax")
])

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
```

### Using Raw Logits

We can also use raw logits instead of applying sigmoid or softmax in the output layer.

For example:

```python
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=True)
```

CCE and SCCE also support `from_logits=True`.

**Important:**
- `from_logits=False` means the loss function receives probabilities.
- `from_logits=True` means the loss function receives raw logits.
- When using `from_logits=True`, do not apply sigmoid or softmax in the output layer.

### Key Takeaways

- Binary classification → BCE + Sigmoid.
- Multiclass with one-hot labels → CCE + Softmax.
- Multiclass with integer labels → SCCE + Softmax.
- `from_logits=True` is used when the model outputs raw logits instead of probabilities.